# Detect incorrectly answered non-RAG questions

Choose a prediction CSV below. This notebook compares its `answer` values with the dataset's `correct_answer` values and lists every incorrect `question_id` in dataset order. Partial result files are supported.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_PATH = PROJECT_ROOT / "data" / "cancermyth_screening_dataset.json"

# Change this filename to inspect another GPT-5.6 Luna non-RAG result file.
ANSWERS_PATH = PROJECT_ROOT / "non_rag" / "gpt-5.6-luna" / "answers_oncology_expert.csv"

print(f"Dataset: {DATASET_PATH}")
print(f"Answers: {ANSWERS_PATH}")

In [ ]:
def parse_boolean(value: str, *, question_id: str) -> bool:
    normalized = value.strip().lower()
    if normalized == "true":
        return True
    if normalized == "false":
        return False
    raise ValueError(f"Answer for question_id={question_id} must be true or false; got {value!r}.")


with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    dataset = json.load(dataset_file)

if not isinstance(dataset, list) or not dataset:
    raise ValueError("The dataset must be a non-empty JSON array.")

dataset_by_id: dict[str, dict] = {}
for record in dataset:
    question_id = str(record.get("id"))
    if question_id in dataset_by_id:
        raise ValueError(f"Duplicate question ID in dataset: {question_id}")
    if not isinstance(record.get("correct_answer"), bool):
        raise ValueError(f"correct_answer for question_id={question_id} must be Boolean.")
    dataset_by_id[question_id] = record

if not ANSWERS_PATH.is_file():
    raise FileNotFoundError(f"Answer file does not exist: {ANSWERS_PATH}")

predictions: dict[str, bool] = {}
with ANSWERS_PATH.open(newline="", encoding="utf-8") as answers_file:
    reader = csv.DictReader(answers_file)
    if reader.fieldnames != ["question_id", "answer"]:
        raise ValueError("The answer CSV must contain exactly: question_id, answer")
    for row in reader:
        question_id = row["question_id"].strip()
        if question_id not in dataset_by_id:
            raise ValueError(f"Unknown question_id in answer CSV: {question_id}")
        if question_id in predictions:
            raise ValueError(f"Duplicate question_id in answer CSV: {question_id}")
        predictions[question_id] = parse_boolean(row["answer"], question_id=question_id)

print(f"Loaded {len(predictions):,} predictions from {ANSWERS_PATH.name}.")

In [ ]:
wrong_question_ids = [
    record["id"]
    for record in dataset
    if str(record["id"]) in predictions
    and predictions[str(record["id"])] != record["correct_answer"]
]

print(f"Wrong answers: {len(wrong_question_ids):,} / {len(predictions):,}")
print("Wrong question IDs:")
wrong_question_ids